In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
# venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_dir = f'/rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/virtual_envs/SurvivEHR-3.10.4'

venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

%load_ext autoreload
%autoreload 2

print(os.getcwd())

Added path '/rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/virtual_envs/SurvivEHR-3.10.4/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/modelling/SurvivEHR/notebooks/CompetingRisk/3_Regional_analaysis


# Load results

In [ ]:
def iter_pickles(dir_path, exts=(".pkl", ".pickle")):
    """Yield (Path, obj) for every pickle in dir_path (recursively)."""
    for p in sorted(Path(dir_path).rglob("*")):
        if p.suffix.lower() in exts:
            with p.open("rb") as f:
                yield p, pickle.load(f)

data = []
for path, obj in iter_pickles("results"):
    # ensure plain dict
    d = dict(obj)
    # record provenance
    d["_source_file"] = path.name       
    # store
    data.append(pd.DataFrame(d))

# Combine
data = pd.concat(data, ignore_index=True)

# Rename W&B log names
# Metrics
rename_map_metrics = {
    "Test:OutcomePerformanceMetricsctd": "Ctd",
    "Test:OutcomePerformanceMetricsibs": "IBS",
    "Test:OutcomePerformanceMetricsinbll": "INBLL",
    "test_loss_desurv": "test_loss"
    }
data["Metric"] = data["Metric"].map(rename_map_metrics)
# Pre-trained
rename_map_pre_trained = {
    "from-scratch": "SFT",
    "SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1": "FFT",
    }
data["Pre-trained"] = data["Pre-trained"].map(rename_map_pre_trained)


display(data.head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def barplot_by_eval_pop(dataframe, task="Hypertension", by_metric="Ctd"):

    plt.close()
    
    dataframe = dataframe[dataframe["Metric"] == by_metric]
    dataframe = dataframe[dataframe["Task"] == task]

    # Combine training scheme columns
    dataframe["Training schema"] = (
        dataframe["Pre-trained"].astype(str) + " " + dataframe["Training sub-population"].astype(str)
    )

    pops = sorted(dataframe["Training sub-population"].unique())
    pre_trains = sorted(dataframe["Pre-trained"].unique(), reverse=True)
    hue_order = np.concatenate([[pre + " " + pop for pre in pre_trains] for pop in pops])
    
    ax = sns.barplot(
        data=dataframe,
        x="Evaluation sub-population",
        y="Metric value",
        hue="Training schema",
        hue_order=hue_order,
    )

    plt.ylabel(by_metric)
    
    # Remove legend
    if ax.legend_:
        ax.legend_.remove()

    # Put hue labels inside each bar, rotated (sideways)
    for container, label in zip(ax.containers, hue_order):
        ax.bar_label(
            container,
            labels=[label] * len(container),
            label_type="center",     # center of the bar
            rotation=90,             # sideways
            color="white",           # visible on filled bars
            fontsize=12,
            padding=0
        )

    plt.title("Performance Across Regions (Ctd)")
    plt.ylabel(by_metric)

    Path("results/barplot_by_eval_pop").mkdir(exist_ok=True)
    plt.savefig(f"results/barplot_by_eval_pop/{task}_{by_metric}.png")



for task in ["Hypertension", "CVD", "MM"]:
    for metric in list(rename_map_metrics.values()):
        barplot_by_eval_pop(data, task=task, by_metric=metric)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def barplot_by_schema(dataframe, task="Hypertension", by_metric="Ctd"):

    plt.close()
    
    dataframe = dataframe[dataframe["Metric"] == by_metric]
    dataframe = dataframe[dataframe["Task"] == task]

    # Combine training scheme columns
    dataframe["Training schema"] = (
        dataframe["Pre-trained"].astype(str) + " " + dataframe["Training sub-population"].astype(str)
    )

    pops = sorted(dataframe["Training sub-population"].unique())
    pre_trains = sorted(dataframe["Pre-trained"].unique(), reverse=True)
    hue_order = np.concatenate([[pre + " " + pop for pre in pre_trains] for pop in pops])
    
    ax = sns.barplot(
        data=dataframe,
        x="Training schema",
        y="Metric value",
        hue="Evaluation sub-population",
        # hue_order=hue_order,
        order=hue_order
    )

    plt.xticks(rotation=45)
    plt.ylabel(by_metric)

    leg = ax.legend(title="Evaluation population", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

    plt.title(f"Performance Across Regions ({task})")  
    plt.ylabel(by_metric)

    plt.tight_layout()

    Path("results/barplot_by_schema").mkdir(exist_ok=True)
    plt.savefig(f"results/barplot_by_schema/{task}_{by_metric}.png")



for task in ["Hypertension", "CVD", "MM"]:
    for metric in list(rename_map_metrics.values()):
        barplot_by_schema(data, task=task, by_metric=metric)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def barplot_by_improvement(
    df,
    task="Hypertension", 
    by_metrics=["Ctd"], 
    y_axis="Absolute"):

    df = df[df["Task"] == task]
    df["Metric value improvement"] = None
    
    plt.close()
    fig, axes = plt.subplots(len(by_metrics), 2,
                             figsize=(5,8), 
                             sharex=True,
                             sharey=True if y_axis == "Percentage" else "row", 
                             )
    for idx, by_metric in enumerate(by_metrics):
    
        dataframe = df[df["Metric"] == by_metric]
    
        # Combine training scheme columns
        # dataframe["Training schema"] = (
        #     dataframe["Pre-trained"].astype(str) + " " + dataframe["Training sub-population"].astype(str)
        # )
    
        for row_idx, row in dataframe.iterrows():
            
            if row["Pre-trained"] == "FFT":
                
                # Get corresponding SFT result
                corresponding_SFT = (
                    (dataframe["Task"] == row["Task"]) &
                    (dataframe["Training sub-population"] == row["Training sub-population"]) &
                    (dataframe["Evaluation sub-population"] == row["Evaluation sub-population"]) &
                    (dataframe["Metric"] == row["Metric"]) &
                    (dataframe['Seed'] == row["Seed"]) &
                    (dataframe["Pre-trained"] == "SFT")
                    )
    
                baseline_SFT_performance = dataframe[corresponding_SFT]["Metric value"]
                assert len(baseline_SFT_performance) == 1, dataframe
    
                delta_improvement = row["Metric value"] - baseline_SFT_performance.item()
                percentage_improvement = ((row["Metric value"] / baseline_SFT_performance.item()) - 1) * 100
                if by_metric in ["IBS", "INBLL", "test_loss"]:
                    delta_improvement *= -1
                    percentage_improvement *= -1
    
                if y_axis == "Absolute":
                    dataframe.loc[row_idx, "Metric value improvement"] = delta_improvement
                elif y_axis == "Percentage":
                    dataframe.loc[row_idx, "Metric value improvement"] = percentage_improvement
    
        dataframe = dataframe[dataframe["Pre-trained"] == "FFT"]
        dataframe_in = dataframe[dataframe["Training sub-population"] == dataframe["Evaluation sub-population"]]
        dataframe_out = dataframe[dataframe["Training sub-population"] != dataframe["Evaluation sub-population"]]


        order = ["London", "North East"]
        base = sns.color_palette("tab10", n_colors=len(order))
        pal1 = dict(zip(order, base))
        pal2 = dict(zip(order, base[::-1]))

        ax1 = sns.barplot(
            ax=axes[idx, 0],
            data=dataframe_in,
            x="Training sub-population",
            y="Metric value improvement",
            order=order,
            palette=pal1,
        )
    
        ax2 = sns.barplot(
            ax=axes[idx, 1],
            data=dataframe_out,
            x="Training sub-population",
            y="Metric value improvement",
            order=order,
            palette=pal2,
        )

        for ax in [ax1]:
            if ax.legend_:
                ax.legend_.remove()
        # leg = ax2.legend(title="Evaluation population", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)


        # After creating ax1, ax2 for this row:
        if idx == 0:  # only annotate the top row
            ax1.text(
                0.5, 1.09, "In-distribution",
                transform=ax1.transAxes, ha="center", va="top",
                fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8)
            )
            ax2.text(
                0.5, 1.09, "Out-of-distribution",
                transform=ax2.transAxes, ha="center", va="top",
                fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8)
            )
        # plt.xticks(rotation=45)
        # plt.xlabel("Available population")

        ax1.set_xlabel('')
        ax2.set_xlabel('')
        ax1.set_ylabel(f"{by_metric}")
        ax2.set_ylabel('')

    # ax1.set_xlabel("(eval in-distribution)")
    # ax2.set_xlabel("(eval out-of-distribution)")

    fig.suptitle(f"{task}", fontweight='bold')  
    fig.supxlabel("Available training population")
    fig.supylabel(f"{y_axis} benefit of pre-training (scratch vs. FFT)")
    
    plt.tight_layout()

    Path("results/barplot_by_improvement").mkdir(exist_ok=True)
    plt.savefig(f"results/barplot_by_improvement/{task}.png")



for task in ["Hypertension", "CVD", "MM"]:
    barplot_by_improvement(data, task=task, by_metrics=list(rename_map_metrics.values())[:-1], y_axis="Absolute")


In [ ]:
tmp_data = data[
    (data["Task"]=="MM") &
    (data["Training sub-population"]=="North East") &
    (data["Evaluation sub-population"]=="London") &
    (data["Metric"]=="INBLL")
    ]

print(tmp_data.sort_values(["Training sub-population", "Pre-trained"]))

In [ ]:
def get_table(dataframe, task="Hypertension", by_metric="Ctd"):

    dataframe = dataframe[dataframe["Metric"] == by_metric]
    dataframe = dataframe[dataframe["Task"] == task]
    
    dataframe["Training sub-population"] = (
        dataframe["Training sub-population"].astype(str) + " " + dataframe["Pre-trained"].astype(str)
    )

    reduced_df = dataframe
    display(reduced_df)
    
    # group by training/evaluation sub-pop and aggregate
    summary = (
        reduced_df
        .groupby(["Training sub-population", "Evaluation sub-population"])["Metric value"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(3).astype(str) #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

table_ctd = get_table(data, by_metric="Ctd")
print(table_ctd)

# Results

In [ ]:
def get_table(dataframe, by_metric="Ctd"):

    print(dataframe.keys())
    data_metric = dataframe[dataframe["Metric"] == by_metric]

    # group by training/evaluation sub-pop and aggregate
    summary = (
        data_metric
        .groupby(["Training sub-population", "Evaluation sub-population"])["Metric value"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(3).astype(str) #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

In [ ]:
table_ctd = get_table(data, by_metric="Ctd")
print(table_ctd)

In [ ]:
table_ibs = get_table(data, by_metric="IBS")
print(table_ibs)

In [ ]:
table_inbll = get_table(data, by_metric="INBLL")
print(table_inbll)

# Results (scaled)

We now scale the results to see relative differences. Using the equation 

$$\frac{Eval(Y \mid P, X)}{Eval(X \mid P=0, X)}$$

where Y is the evaluation population, trained on sub-population X, with P a binary indicator denoting whether we intiialised from a pre-trained model (1), or not (0).

In [ ]:
def get_scaled_table(df, by_metric="Ctd"):

    # Get the dataset type
    dataset = df["Task"][0]
    
    df_metric = df.loc[df["Metric"] == by_metric].copy()
    df_metric["Scaled metric change"] = np.nan

    for idx, row in df_metric.iterrows():
        seed = row["Seed"]
        training_full = row["Training sub-population"]
        training_base = training_full.split(",")[0]
        row_value = row["Metric value"]

        # scaler
        scaler_row = df_metric[
            (df_metric["Seed"] == seed) &
            (df_metric["Training sub-population"] == training_base + ", P=0") &
            (df_metric["Evaluation sub-population"] == training_base.replace(" ", "-"))
        ]
        scaler = scaler_row["Metric value"].iloc[0]

        df_metric.at[idx, "Scaled metric change"] = ((row_value / scaler) - 1) * 100

    # group by training/evaluation sub-pop and aggregate
    summary = (
        df_metric
        .groupby(["Training sub-population", "Evaluation sub-population"])["Scaled metric change"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(2).astype(str) + "%" #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

In [ ]:
scaled_table_ctd = get_scaled_table(data, by_metric="Ctd")
print(scaled_table_ctd)

In [ ]:
scaled_table_ibs = get_scaled_table(data, by_metric="IBS")
print(scaled_table_ibs)

In [ ]:
scaled_table_inbll = get_scaled_table(data, by_metric="INBLL")
print(scaled_table_inbll)

# Results (scaled)

We now scale the results to see relative differences, but do not assume an model that was not pre-trained as the baseline. Using the equation 

$$\frac{Eval(Y \mid P, X)}{Eval(X \mid P, X)}$$

where Y is the evaluation population, trained on sub-population X, with P a binary indicator denoting whether we intiialised from a pre-trained model (1), or not (0).

In [ ]:
def get_scaled2_table(df, by_metric="Ctd"):

    # Get the dataset type
    dataset = df["Task"][0]
    
    df_metric = df.loc[df["Metric"] == by_metric].copy()
    df_metric["Scaled metric change"] = np.nan

    for idx, row in df_metric.iterrows():
        seed = row["Seed"]
        training_full = row["Training sub-population"]
        training_base = training_full.split(",")[0]
        row_value = row["Metric value"]

        # scaler
        scaler_row = df_metric[
            (df_metric["Seed"] == seed) &
            (df_metric["Training sub-population"] == training_full) &
            (df_metric["Evaluation sub-population"] == training_base.replace(" ", "-"))
        ]
        scaler = scaler_row["Metric value"].iloc[0]

        df_metric.at[idx, "Scaled metric change"] = ((row_value / scaler) - 1) * 100

    # group by training/evaluation sub-pop and aggregate
    summary = (
        df_metric
        .groupby(["Training sub-population", "Evaluation sub-population"])["Scaled metric change"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(2).astype(str) + "%" #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

In [ ]:
scaled_table_ctd = get_scaled2_table(data, by_metric="Ctd")
print(scaled_table_ctd)

In [ ]:
scaled_table_ibs = get_scaled2_table(data, by_metric="IBS")
print(scaled_table_ibs)

In [ ]:
scaled_table_inbll = get_scaled2_table(data, by_metric="INBLL")
print(scaled_table_inbll)